In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
# Create the SparkSession
spark = SparkSession\
        .builder\
        .appName("sparkstream")\
        .getOrCreate()

In [3]:

# Define input sources
lines = (spark\
        .readStream.format("socket")\
        .option("host", "localhost")\
        .option("port", 9999)\
        .load())

In [4]:
# Transform data
words = lines.select(split(col("value"), "\\s").alias("word"))

In [5]:
# Get the count of published words
counts = words.groupBy("word").count()

In [6]:
# Define the checkpoint directory
checkpointDir = "C:/Users/Admin/Documents/Udemy/Microsoft Azure Data Engineer/Spark Tutorial & Practice/checkpoint/"

In [7]:
# Start streaming defining the necessary configurations
streamingQuery = (counts
.writeStream
.format("console")
.outputMode("complete")
.trigger(processingTime="1 second")
.option("checkpointLocation", checkpointDir)
.start())
streamingQuery.awaitTermination()

StreamingQueryException: [STREAM_FAILED] Query [id = 276685c3-088b-4468-9fb6-b368255b2ad1, runId = 453c0b01-6ae2-4833-9678-587d233d2338] terminated with exception: org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Ljava/lang/String;I)Z